# 申万行业拆分与聚类分析

**基于申万一级行业指数数据进行分析**

本notebook使用申万一级行业指数数据，实现以下分析：
1. **行业相关性分析**：计算行业间的相关系数矩阵
2. **行业聚类分析**：基于收益率相关性进行聚类
3. **最大生成树**：构建行业关联网络
4. **可视化分析**：展示聚类结果和行业关系网络

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from source.data_fetcher import get_sw_industry_returns, SW_INDUSTRIES
from source import config

print("模块导入成功！")

## 1. 数据加载

加载本地保存的申万行业收益率数据

In [ ]:
# 数据配置
START_DATE = config.DATA_CONFIG['start_date']
END_DATE = config.DATA_CONFIG['end_date']

print(f"数据时间范围: {START_DATE} - {END_DATE}")
print(f"\n申万一级行业列表 ({len(SW_INDUSTRIES)} 个行业):")
for industry in SW_INDUSTRIES:
    print(f"  - {industry}")

In [ ]:
# 加载申万行业收益率数据
print("正在加载申万行业收益率数据...")

industry_returns = get_sw_industry_returns()

print(f"\n行业收益率矩阵形状: {industry_returns.shape}")
print(f"时间范围: {industry_returns.index[0]} 至 {industry_returns.index[-1]}")
print(f"有效行业数: {industry_returns.shape[1]}")

# 去除全为空值的列
industry_returns = industry_returns.dropna(axis=1, how='all')
print(f"去除空列后行业数: {industry_returns.shape[1]}")

industry_returns.head()

## 2. 行业相关性分析

计算行业间的收益率相关系数矩阵

In [ ]:
# 计算相关系数矩阵
corr_matrix = industry_returns.corr()

print(f"相关系数矩阵形状: {corr_matrix.shape}")
print("\n相关系数矩阵 (前10个行业):")
corr_matrix.iloc[:10, :10]

In [ ]:
# 绘制相关系数热力图
plt.figure(figsize=(14, 12))
plt.imshow(corr_matrix, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(label='Correlation Coefficient')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90, fontsize=8)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index, fontsize=8)
plt.title('Industry Return Correlation Matrix (Shenwan Level-1)', fontsize=14)
plt.tight_layout()

# 保存图片
os.makedirs(config.FILE_PATHS['output'], exist_ok=True)
plt.savefig(os.path.join(config.FILE_PATHS['output'], 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n相关系数热力图已保存至: {config.FILE_PATHS['output']}correlation_heatmap.png")

## 3. 行业聚类分析

基于蒙特卡洛K-means算法对行业进行聚类

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

# 将相关系数转换为距离矩阵
distance_matrix = 1 - corr_matrix

# 确保对角线为0
np.fill_diagonal(distance_matrix.values, 0)

# 转换为压缩距离矩阵
condensed_distance = squareform(distance_matrix.values, checks=False)

# 进行层次聚类
linkage_matrix = linkage(condensed_distance, method='ward')

print("层次聚类完成！")

In [ ]:
# 绘制树状图
plt.figure(figsize=(16, 10))
dendrogram(
    linkage_matrix,
    labels=corr_matrix.columns.tolist(),
    leaf_rotation=90,
    leaf_font_size=10
)
plt.title('Hierarchical Clustering Dendrogram (Shenwan Industries)', fontsize=14)
plt.xlabel('Industry', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(config.FILE_PATHS['output'], 'clustering_dendrogram.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n聚类树状图已保存至: {config.FILE_PATHS['output']}clustering_dendrogram.png")

In [ ]:
# 根据距离阈值进行聚类
n_clusters = config.DATA_CONFIG['n_clusters']
cluster_labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')

# 创建聚类结果DataFrame
cluster_df = pd.DataFrame({
    'industry': corr_matrix.columns,
    'cluster': cluster_labels
})

print(f"聚类数量: {n_clusters}")
print("\n聚类结果:")
for i in range(1, n_clusters + 1):
    industries = cluster_df[cluster_df['cluster'] == i]['industry'].tolist()
    print(f"\nCluster {i} ({len(industries)} 个行业):")
    print(f"  {', '.join(industries)}")

## 4. 最大生成树分析

基于相关系数构建最大生成树，展示行业间的核心关联关系

In [ ]:
import networkx as nx

# 创建全连接图
G_full = nx.Graph()
G_full.add_nodes_from(corr_matrix.columns)

# 添加边和权重（相关系数）
for i, ind1 in enumerate(corr_matrix.columns):
    for j, ind2 in enumerate(corr_matrix.columns):
        if i < j:
            corr = corr_matrix.loc[ind1, ind2]
            if not np.isnan(corr):
                G_full.add_edge(ind1, ind2, weight=corr)

# 计算最大生成树（使用相关系数的相反数作为距离）
G_mst = nx.maximum_spanning_tree(G_full)

print(f"最大生成树节点数: {G_mst.number_of_nodes()}")
print(f"最大生成树边数: {G_mst.number_of_edges()}")

# 获取边信息
mst_edges = []
for u, v, d in G_mst.edges(data=True):
    mst_edges.append({
        'industry1': u,
        'industry2': v,
        'correlation': d['weight']
    })

mst_df = pd.DataFrame(mst_edges).sort_values('correlation', ascending=False)
print("\n最大生成树的主要边（按相关系数排序）:")
mst_df.head(15)

In [ ]:
# 绘制最大生成树
plt.figure(figsize=(16, 12))

# 使用spring布局
pos = nx.spring_layout(G_mst, k=2, iterations=50, seed=42)

# 获取边权重
edges = G_mst.edges()
weights = [G_mst[u][v]['weight'] for u, v in edges]

# 绘制节点
node_colors = [cluster_df[cluster_df['industry'] == node]['cluster'].values[0] for node in G_mst.nodes()]
nx.draw_networkx_nodes(G_mst, pos, node_color=node_colors, node_size=800, cmap=plt.cm.Set1, alpha=0.9)

# 绘制边
nx.draw_networkx_edges(G_mst, pos, edge_color=weights, edge_cmap=plt.cm.RdYlBu_r,
                       width=2, alpha=0.8)

# 绘制标签
nx.draw_networkx_labels(G_mst, pos, font_size=8, font_family='sans-serif')

# 添加边标签
edge_labels = {(u, v): f'{d["weight"]:.2f}' for u, v, d in G_mst.edges(data=True)}
nx.draw_networkx_edge_labels(G_mst, pos, edge_labels=edge_labels, font_size=6)

plt.title('Maximum Spanning Tree of Shenwan Industries', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(config.FILE_PATHS['output'], 'maximum_spanning_tree.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n最大生成树图已保存至: {config.FILE_PATHS['output']}maximum_spanning_tree.png")

## 5. 聚类结果汇总

汇总各聚类的行业组成和特征

In [ ]:
# 计算每个聚类的平均相关系数（内部）
cluster_summary = []

for cluster_id in range(1, n_clusters + 1):
    industries = cluster_df[cluster_df['cluster'] == cluster_id]['industry'].tolist()
    
    if len(industries) > 1:
        # 计算内部平均相关系数
        internal_corrs = []
        for i, ind1 in enumerate(industries):
            for j, ind2 in enumerate(industries):
                if i < j:
                    corr = corr_matrix.loc[ind1, ind2]
                    if not np.isnan(corr):
                        internal_corrs.append(corr)
        internal_corr = np.mean(internal_corrs) if internal_corrs else 0
    else:
        internal_corr = 1.0
    
    cluster_summary.append({
        'cluster': cluster_id,
        'n_industries': len(industries),
        'industries': ', '.join(industries),
        'internal_avg_corr': internal_corr
    })

cluster_summary_df = pd.DataFrame(cluster_summary)
cluster_summary_df = cluster_summary_df.sort_values('internal_avg_corr', ascending=False)

print("聚类汇总表:")
print("=" * 80)
for _, row in cluster_summary_df.iterrows():
    print(f"\n聚类 {row['cluster']} ({row['n_industries']} 个行业, 内部平均相关系数: {row['internal_avg_corr']:.3f})")
    print(f"  行业: {row['industries']}")

In [ ]:
# 保存聚类结果到CSV
cluster_df.to_csv(os.path.join(config.FILE_PATHS['output'], 'cluster_result.csv'), index=False)
mst_df.to_csv(os.path.join(config.FILE_PATHS['output'], 'mst_edges.csv'), index=False)
corr_matrix.to_csv(os.path.join(config.FILE_PATHS['output'], 'correlation_matrix.csv'))
cluster_summary_df.to_csv(os.path.join(config.FILE_PATHS['output'], 'cluster_summary.csv'), index=False)

print("结果文件已保存:")
print(f"  - {config.FILE_PATHS['output']}cluster_result.csv")
print(f"  - {config.FILE_PATHS['output']}mst_edges.csv")
print(f"  - {config.FILE_PATHS['output']}correlation_matrix.csv")
print(f"  - {config.FILE_PATHS['output']}cluster_summary.csv")

## 6. 分析结论

基于申万一级行业指数的聚类分析，得出以下结论：

1. **高相关性行业群**：
   - 银行与非银金融相关性较高
   - 食品饮料与医药生物存在一定相关性
   - 电子、计算机、传媒、通信（TMT）形成明显的高相关群

2. **低相关性行业**:
   - 煤炭、石油石化等能源行业与其他行业相关性较低
   - 银行、保险等金融行业与TMT行业相关性较低

3. **最大生成树核心边**：
   - 揭示了行业间最核心的关联关系
   - 可用于理解行业轮动和风险传导